# Data Extraction

These notebooks uses publically available company accounts from Companies House. 
The code here is a quick and simple method to extract the key feature(description) and label(xbrl concept). It is sufficient for to demonstrate the core of the ML methodology used 


In [ ]:
import os
import re
from pathlib import Path

from bs4 import BeautifulSoup
import pandas as pd


# 1. Extraction functions

Extracts iXBRL concept name as value, and first cell value as description.


In [ ]:
def extract_xbrl_tags(row_element):
    
    tags = []
    for cell in row_element.find_all(['td', 'th']):
        xbrl_tags = cell.find_all(['ix:nonnumeric', 'ix:nonfraction'])
        for tag in xbrl_tags:
            tags.append({
                'tag_name': re.sub(r'^.*:', '', tag.get('name', '')),
                'value': tag.text.strip(),
                'context_ref': tag.get('contextref', '')
            })
    return tags


def analyze_table(table_soup, table_index):

    extracted_tables = pd.DataFrame()
    for row in table_soup.find_all('tr'):
        first_cell = row.find(['td', 'th'])
        if first_cell:
            description = first_cell.text.strip()
            if description and description != 'nan':
                xbrl_tags = extract_xbrl_tags(row)
                if xbrl_tags:
                    table_row = pd.DataFrame([{'table_index' : table_index, 'description' : description, 'label' : xbrl_tags[0]['tag_name']}])
                    extracted_tables = pd.concat([extracted_tables, table_row])


    return extracted_tables

def extract_all_tables(file_path):

    # print(f"extracting {file_path}")
    
    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
    html_content = None
    
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                html_content = f.read()
            break
        except UnicodeDecodeError:
            continue
    
    if html_content is None:
        print(f"Warning: Could not read file {file_path}")
        return pd.DataFrame()

    soup = BeautifulSoup(html_content, 'html.parser')
    tables = soup.find_all('table')

    all_tables_df = pd.DataFrame()
    for table_index in range(1, len(tables)):
        all_tables_df = pd.concat([all_tables_df, analyze_table(tables[table_index], table_index)])
    return all_tables_df


# 2. Extract and Save
Data saved from https://download.companieshouse.gov.uk/Accounts_Monthly_Data-November2025.zip
298,461 Accounts
This is a lot of data and takes up a lot of space(18GB) and takes a long time to process

In [ ]:
path_to_data = Path('~/Downloads/Accounts_Monthly_Data-November2025').expanduser()
files = os.listdir(path_to_data)
path_to_data
files_paths = [str(path_to_data) + '/' + file for file in files]
table_extracted_lst = map(extract_all_tables, files_paths)
table_extracted_df = pd.concat(table_extracted_lst, ignore_index=True)

table_extracted_df.to_parquet('data/table_extracted_df_v5.parquet')